In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

from tqdm import tqdm
import matplotlib.pyplot as plt 
import neurobayes as nb
import numpy as np

c:\Users\Acer\miniconda3\envs\bayes\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Preparing the dataset and creating the features and target arrays

n_sims = 1000

x = []
y = []

for i in tqdm(range(n_sims), desc='Carregando dados', ascii=True):
    data = np.load(f'../input/data_{i+1}.npy')
    x.append(data[:, :2])
    y.append(data[0, 2])  # equivalente a data[:, 2:][0][0]

x = np.array(x)
y = np.array(y)

norm = y.max()
# Normalização
# y /= y.max()


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.25)

Carregando dados: 100%|##########| 1000/1000 [00:07<00:00, 126.03it/s]


In [12]:

X_train_flat = x_train.reshape(x_train.shape[0], -1)
X_val_flat = x_test.reshape(x_test.shape[0], -1)


architecture = nb.FlaxMLP(
    hidden_dims=[16, 16],
    target_dim=1,
    # hidden_dropout=0.1
)


bnn = nb.BNN(architecture)



In [23]:

bnn.fit(
    X_train_flat, y_train,
    num_warmup=100,
    num_samples=2000
)

sample: 100%|██████████| 2100/2100 [01:12<00:00, 28.98it/s, 19 steps of size 4.05e-05. acc. prob=0.09] 
Using best run (acceptance rate: 0.094)


In [24]:

mean_pred, var_pred = bnn.predict(X_val_flat)
std_pred = np.sqrt(var_pred)

for i in range(5):
    print(f"Predição: {mean_pred[i][0]:.2f} ± {std_pred[i][0]:.2f}")

Predição: 74.39 ± 0.68
Predição: 72.33 ± 0.68
Predição: 67.61 ± 0.69
Predição: 75.33 ± 0.69
Predição: 72.57 ± 0.68


In [25]:
# forecasting "real" H(0)

real = np.load('../input/data_real80.npy')
real = real[real[:, 1].argsort()]

real = real.reshape(-1,160)

In [26]:
mean_pred, var_pred = bnn.predict(real)
std_pred = np.sqrt(var_pred)


print(f"Predição: {mean_pred[0][0]:.2f} ± {std_pred[0][0]:.2f}")

Predição: 67.51 ± 0.66


In [27]:
mean_pred, var_pred

(Array([[67.509674]], dtype=float32), Array([[0.44052374]], dtype=float32))